# 3. Full Ukrainian generation

Use the full taxonomy matrix, the selected Lapa model, sequential prompting and
the final prompts revised after pilot feedback. Preserve matrix IDs and the
stereotype → counter-stereotype → unrelated dependencies.

Full Ukrainian annotation is in progress. The input matrix and request preview
are available independently of the final annotated corpus.

In [ ]:
import json
import tempfile
from pathlib import Path

from IPython.display import display
from pandas import DataFrame

from fairforget.dataset_workflow import find_project, load_inputs

project = find_project()
inputs = load_inputs(project)
protocol = json.loads((project / "protocol/construction.json").read_text())
work = Path(tempfile.mkdtemp(prefix="warbias-dataset-"))

In [ ]:
from fairforget.warbias.pilot_analysis import describe_matrix, generation_cases

matrix = inputs["matrix_full"]
cases = generation_cases(matrix)
display(
    DataFrame(matrix)[
        [
            "matrix_item_id",
            "category_uk",
            "target_uk",
            "gender_track",
            "age_track",
            "stereotype_claim_uk",
        ]
    ].head(12)
)
display(DataFrame(protocol["prompt_bundle"]["prompts"])[["stage", "path", "sha256"]])
{
    "matrix": describe_matrix(matrix),
    "selected_condition": protocol["selected_condition"],
    "request_settings": protocol["request_settings"],
    "execution_settings": protocol["execution_settings"],
}

In [ ]:
from fairforget.dataset_workflow import recorded_selection, run_full_generation

SELECTION_RECORD_PATH = None
analysis, selection_record = recorded_selection(project, inputs)
if SELECTION_RECORD_PATH is not None:
    selection_record = json.loads(Path(SELECTION_RECORD_PATH).read_text())
preview = run_full_generation(project, matrix, selection_record, work / "full-preview")
first_stage = json.loads((work / "full-preview/generation/S/plan.json").read_text())
display(DataFrame(first_stage["requests"])[["id", "stage", "model", "prompt_sha256"]].head())
preview

The preview prepares the first stage. Later stages require its actual output.
The selected condition is checked against the same ratings and recorded author
choice used in notebook 2. A new execution records that evidence, final prompt
hashes and every generated row in its production manifest.

Set `SELECTION_RECORD_PATH` to the file shown in notebook 2 to pass its saved
choice directly. Leave it unset to load the same choice from the study records.

In [ ]:
RUN_GENERATION = False
PROVIDER_FACTORY = None

result = {"stage": "full_ukrainian_generation", **preview}
if RUN_GENERATION:
    from fairforget.cli_support import load_factory

    if not PROVIDER_FACTORY:
        raise ValueError("Set PROVIDER_FACTORY to module:callable")
    provider_config = {
        key: protocol["selected_condition"][key]
        for key in ("candidate_id", "provider", "model", "revision")
    }
    provider_config["settings"] = protocol["request_settings"]
    provider = load_factory(PROVIDER_FACTORY)(config=provider_config)
    result = {
        "stage": "full_ukrainian_generation",
        **run_full_generation(
            project,
            matrix,
            selection_record,
            work / "full",
            provider=provider,
            execute=True,
        ),
    }
result